# 1 Environment Setup + Data Loading + Evaluation Script



In [1]:
!pip install datasets transformers torch -q
!pip install rank_bm25 -q
print('finish！')

finish！


## Load HotpotQA Dataset

In [2]:
from datasets import load_dataset

print('loading HotpotQA dataset...')
dataset = load_dataset('hotpot_qa', 'distractor')

train_data = dataset['train']
val_data   = dataset['validation']

print(f'train set size: {len(train_data)}')
print(f'validation set size: {len(val_data)}')
print('\nDataset Fields:', train_data.column_names)

loading HotpotQA dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

train set size: 90447
validation set size: 7405

Dataset Fields: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']


In [3]:
import json

sample = val_data[0]

print('【Question】')
print(sample['question'])
print()

print('【Gold Answer】')
print(sample['answer'])
print()

print('【Answer Type / level】')
print(f"type: {sample['type']} | level: {sample['level']}")
print()

print('【Context：】')
for title, sentences in zip(sample['context']['title'], sample['context']['sentences']):
    print(f'  Passage Title: {title}')
    print(f'  Sentence Number: {len(sentences)}')
    print(f'  First Sentence: {sentences[0][:80]}...')
    print()

print('【Supporting Facts：】')
for title, sent_id in zip(sample['supporting_facts']['title'], sample['supporting_facts']['sent_id']):
    print(f'  Passage: {title}, Sentence ID: {sent_id}')

【Question】
Were Scott Derrickson and Ed Wood of the same nationality?

【Gold Answer】
yes

【Answer Type / level】
type: comparison | level: hard

【Context：】
  Passage Title: Ed Wood (film)
  Sentence Number: 3
  First Sentence: Ed Wood is a 1994 American biographical period comedy-drama film directed and pr...

  Passage Title: Scott Derrickson
  Sentence Number: 3
  First Sentence: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and ...

  Passage Title: Woodson, Arkansas
  Sentence Number: 5
  First Sentence: Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the U...

  Passage Title: Tyler Bates
  Sentence Number: 5
  First Sentence: Tyler Bates (born June 5, 1965) is an American musician, music producer, and com...

  Passage Title: Ed Wood
  Sentence Number: 1
  First Sentence: Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American fil...

  Passage Title: Deliver Us from Evil (2014 film)
  Sentence Number: 3

## Format the context into text


In [4]:
def format_context(context, max_paragraphs=10):

    parts = []
    titles = context['title'][:max_paragraphs]
    sentences_list = context['sentences'][:max_paragraphs]

    for title, sentences in zip(titles, sentences_list):
        paragraph_text = ' '.join(sentences)
        parts.append(f'[{title}]\n{paragraph_text}')

    return '\n\n'.join(parts)


def get_supporting_passages(sample):

    support_titles = set(sample['supporting_facts']['title'])
    passages = []

    for title, sentences in zip(
        sample['context']['title'],
        sample['context']['sentences']
    ):
        if title in support_titles:
            passages.append({
                'title': title,
                'text': ' '.join(sentences)
            })

    return passages


# test
print(format_context(sample['context'], max_paragraphs=2))
print()
print('supporting facts：')
for p in get_supporting_passages(sample):
    print(f"  [{p['title']}] {p['text'][:100]}...")

[Ed Wood (film)]
Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.

[Scott Derrickson]
Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."

supporting facts：
  [Scott Derrickson] Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives ...
  [Ed Wood] Edward Davis Wood Jr. (October 10, 1924 – December 10

## EM / F1 Evaluation Function


In [5]:
import re
import string
from collections import Counter


def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def get_tokens(s):
    if not s:
        return []
    return normalize_answer(s).split()


def compute_exact(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))


def compute_f1(prediction, ground_truth):
    pred_tokens  = get_tokens(prediction)
    truth_tokens = get_tokens(ground_truth)
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall    = num_same / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1


def evaluate(predictions, ground_truths):

    assert len(predictions) == len(ground_truths), 'Inconsistent length'

    em_scores = [compute_exact(p, g) for p, g in zip(predictions, ground_truths)]
    f1_scores = [compute_f1(p, g)    for p, g in zip(predictions, ground_truths)]

    return {
        'em':  sum(em_scores) / len(em_scores),
        'f1':  sum(f1_scores) / len(f1_scores),
        'n':   len(predictions)
    }


# test
test_preds  = ['Edward Jenner', 'the united states', 'yes']
test_truths = ['Edward Jenner', 'United States', 'yes']

results = evaluate(test_preds, test_truths)
print(f"Result：EM={results['em']:.3f}, F1={results['f1']:.3f}, N={results['n']}")

Result：EM=1.000, F1=1.000, N=3


## Extract an experimental subset

the validation set has 7,405 entries—running all of them is too slow and expensive. We select 200 samples for experiments while ensuring a balanced distribution across types.

In [6]:
import random
random.seed(42)

# Stratified sampling by type: bridge questions (multi-hop reasoning) and comparison questions.
bridge_samples    = [s for s in val_data if s['type'] == 'bridge']
comparison_samples = [s for s in val_data if s['type'] == 'comparison']

print(f'Validation size: {len(val_data)}')
print(f'Bridge: {len(bridge_samples)}')
print(f'Comparison: {len(comparison_samples)}')

sample_bridge     = random.sample(bridge_samples,     160)
sample_comparison = random.sample(comparison_samples,  40)
eval_samples      = sample_bridge + sample_comparison

random.shuffle(eval_samples)

print(f'\nSubset Size: {len(eval_samples)}')
print('Distribution：', {
    'bridge': sum(1 for s in eval_samples if s['type'] == 'bridge'),
    'comparison': sum(1 for s in eval_samples if s['type'] == 'comparison')
})

Validation size: 7405
Bridge: 5918
Comparison: 1487

Subset Size: 200
Distribution： {'bridge': 160, 'comparison': 40}


In [7]:
import json

with open('eval_samples.json', 'w') as f:
    json.dump(eval_samples, f, ensure_ascii=False, indent=2)

print('Saved in eval_samples.json')

Saved in eval_samples.json


# 2 Naive Baseline


In [8]:
import json
import re
import string
import time
from collections import Counter

with open('eval_samples.json', 'r') as f:
    eval_samples = json.load(f)

print(f'Load {len(eval_samples)} samples')

Load 200 samples


## Load model

In [9]:
# Qwen2.5-7B-Instruct

!pip install transformers accelerate -q
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = 'Qwen/Qwen2.5-7B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map='auto'
)

def call_model(prompt, max_tokens=64):
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.01, do_sample=False)
    gen = output[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## build Naive Baseline Prompt


In [10]:
def build_index_hint(context, max_paragraphs=10):

    parts = []
    for title, sentences in zip(
        context['title'][:max_paragraphs],
        context['sentences'][:max_paragraphs]
    ):
        lines = [f"[{i}] {s}" for i, s in enumerate(sentences)]
        parts.append(f"[{title}]\n" + '\n'.join(lines))
    return '\n\n'.join(parts)


def parse_sp_output(raw_output, valid_titles):

    answer = ''
    sp = []

    ans_match = re.search(r'Answer:\s*(.+?)(?:\n|$)', raw_output)
    if ans_match:
        answer = ans_match.group(1).strip()

    sp_match = re.search(r'Supporting facts:\s*(.+?)(?:\n|$)', raw_output, re.DOTALL)
    if sp_match:
        sp_text = sp_match.group(1).strip()

        pattern = r'\[([^\]]+),\s*(\d+)\]'
        matches = re.findall(pattern, sp_text)
        for title, sent_id in matches:
            title = title.strip()
            if title in valid_titles:
                sp.append([title, int(sent_id)])

    return answer, sp

In [11]:
def build_naive_prompt(sample):
    context_text = format_context(sample['context'], max_paragraphs=10)

    index_text = build_index_hint(sample['context'])

    prompt = f"""Answer the question based on the provided passages.

Output format (strictly follow this):
Answer: <your answer>
Supporting facts: [Title, sentence_index], [Title, sentence_index], ...

Rules:
- Answer should be as short as possible (a few words)
- Supporting facts must use exact titles from the passages
- sentence_index starts from 0

Passages (with sentence indices):
{index_text}

Question: {sample['question']}"""
    return prompt


sample = eval_samples[0]
prompt = build_naive_prompt(sample)
print('===== Prompt=====')
print(prompt[:600])
print('...')

print(f'Gold Answer: {sample["answer"]}')

===== Prompt=====
Answer the question based on the provided passages.

Output format (strictly follow this):
Answer: <your answer>
Supporting facts: [Title, sentence_index], [Title, sentence_index], ...

Rules:
- Answer should be as short as possible (a few words)
- Supporting facts must use exact titles from the passages
- sentence_index starts from 0

Passages (with sentence indices):
[Moe Szyslak]
[0] Morris "Moe" Szyslak is a fictional character from the American animated television series "The Simpsons".
[1]  He is voiced by Hank Azaria and first appeared in the series premiere episode "Simpsons Roasting o
...
Gold Answer: Daniel Louis Castellaneta


In [13]:
import time

naive_predictions   = []
naive_ground_truths = []
naive_records       = []


for i, sample in enumerate(eval_samples):
    prompt       = build_naive_prompt(sample)
    valid_titles = sample['context']['title']

    try:
        raw_output     = call_model(prompt, max_tokens=128)  # 128给sp留空间
        pred, sp_pred  = parse_sp_output(raw_output, valid_titles)
    except Exception as e:
        print(f'  Sample {i} error: {e}')
        raw_output, pred, sp_pred = '', '', []

    naive_predictions.append(pred)
    naive_ground_truths.append(sample['answer'])
    naive_records.append({
        'id':           i,
        'question':     sample['question'],
        'answer':       sample['answer'],
        'prediction':   pred,
        'sp_prediction': sp_pred,
        'raw_output':   raw_output,
        'type':         sample['type'],
        'level':        sample['level'],
        'em':           compute_exact(pred, sample['answer']),
        'f1':           compute_f1(pred, sample['answer'])
    })

    if (i + 1) % 20 == 0:
        partial = evaluate(naive_predictions, naive_ground_truths)
        sp_coverage = sum(1 for r in naive_records if len(r['sp_prediction']) > 0)
        print(f'[{i+1}/200] EM={partial["em"]:.3f}, F1={partial["f1"]:.3f} | sp coverage: {sp_coverage}/{i+1}')

    time.sleep(0.3)

print('\nFinish！')

# 检查sp解析情况
sp_empty = sum(1 for r in naive_records if len(r['sp_prediction']) == 0)
print(f'\nsp Number of samples with parsing failures (empty list): {sp_empty}/200')


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[20/200] EM=0.250, F1=0.358 | sp coverage: 19/20
[40/200] EM=0.325, F1=0.398 | sp coverage: 39/40
[60/200] EM=0.283, F1=0.338 | sp coverage: 58/60
  Sample 76 error: CUDA out of memory. Tried to allocate 1.97 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.48 GiB is free. Including non-PyTorch memory, this process has 13.08 GiB memory in use. Of the allocated memory 12.35 GiB is allocated by PyTorch, and 612.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
[80/200] EM=0.250, F1=0.309 | sp coverage: 77/80
[100/200] EM=0.280, F1=0.327 | sp coverage: 95/100
  Sample 105 error: CUDA out of memory. Tried to allocate 764.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 475.81 MiB is free. Including non-PyTorch memory, this process has 14.10 GiB

## Files required to generate the official evaluation script


In [14]:
def save_for_official_eval(records, output_path):


    prediction = {
        "answer": {},
        "sp": {}
    }
    for r in records:
        qid = str(r['id'])
        prediction["answer"][qid] = r['prediction']
        prediction["sp"][qid]     = r['sp_prediction']

    with open(output_path, 'w') as f:
        json.dump(prediction, f, ensure_ascii=False, indent=2)

    print(f'Saved in {output_path}')
    print(f'answer={len(prediction["answer"])}, sp={len(prediction["sp"])}')


save_for_official_eval(naive_records, 'naive_prediction.json')
# python eval.py naive_prediction.json gold.json

Saved in naive_prediction.json
answer=200, sp=200


## Evaluate directly


In [15]:
import re


def compute_sp_metrics(sp_pred, sp_gold):
    """
    sp_pred: [[title, sent_id], ...]
    sp_gold: [[title, sent_id], ...]
    """
    cur_sp_pred  = set(map(tuple, sp_pred))
    gold_sp_pred = set(map(tuple, sp_gold))
    tp, fp, fn = 0, 0, 0
    for e in cur_sp_pred:
        if e in gold_sp_pred: tp += 1
        else:                  fp += 1
    for e in gold_sp_pred:
        if e not in cur_sp_pred: fn += 1
    prec   = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1     = 2 * prec * recall / (prec + recall) if prec + recall > 0 else 0.0
    em     = 1.0 if fp + fn == 0 else 0.0
    return {'sp_em': em, 'sp_f1': f1, 'sp_prec': prec, 'sp_recall': recall}


def compute_joint_metrics(ans_f1, ans_prec, ans_recall, sp_em, sp_prec, sp_recall, ans_em):
    joint_prec   = ans_prec   * sp_prec
    joint_recall = ans_recall * sp_recall
    joint_f1     = 2 * joint_prec * joint_recall / (joint_prec + joint_recall) if joint_prec + joint_recall > 0 else 0.0
    joint_em     = ans_em * sp_em
    return {'joint_em': joint_em, 'joint_f1': joint_f1,
            'joint_prec': joint_prec, 'joint_recall': joint_recall}



from collections import Counter

def f1_score_full(prediction, ground_truth):
    """return (f1, prec, recall)"""
    def norm(s):
        s = re.sub(r'\b(a|an|the)\b', ' ', s.lower())
        s = ''.join(ch for ch in s if ch not in set(string.punctuation))
        return ' '.join(s.split())

    if norm(prediction) in ['yes','no','noanswer'] and norm(prediction) != norm(ground_truth):
        return 0.0, 0.0, 0.0
    if norm(ground_truth) in ['yes','no','noanswer'] and norm(prediction) != norm(ground_truth):
        return 0.0, 0.0, 0.0

    p_toks = norm(prediction).split()
    g_toks = norm(ground_truth).split()
    common = Counter(p_toks) & Counter(g_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0, 0.0, 0.0
    prec   = num_same / len(p_toks)
    recall = num_same / len(g_toks)
    f1     = 2 * prec * recall / (prec + recall)
    return f1, prec, recall


In [16]:
for r in naive_records:

    idx      = r['id']
    sample   = eval_samples[idx]
    sp_gold  = list(zip(
        sample['supporting_facts']['title'],
        sample['supporting_facts']['sent_id']
    ))

    ans_f1, ans_prec, ans_recall = f1_score_full(r['prediction'], r['answer'])
    ans_em = compute_exact(r['prediction'], r['answer'])

    # sp
    sp_metrics = compute_sp_metrics(r['sp_prediction'], sp_gold)

    # joint
    joint_metrics = compute_joint_metrics(
        ans_f1, ans_prec, ans_recall,
        sp_metrics['sp_em'], sp_metrics['sp_prec'], sp_metrics['sp_recall'],
        ans_em
    )

    r.update(sp_metrics)
    r.update(joint_metrics)


In [17]:

def avg(records, key):
    return sum(r[key] for r in records) / len(records)

naive_results = evaluate(naive_predictions, naive_ground_truths)

print('=' * 52)
print('Naive Baseline Result')
print('=' * 52)
print(f"{'Metrics':<16} {'EM':>8} {'F1':>8}")
print('-' * 36)
print(f"{'ans':<16} {avg(naive_records,'em'):>8.4f} {avg(naive_records,'f1'):>8.4f}")
print(f"{'sp':<16} {avg(naive_records,'sp_em'):>8.4f} {avg(naive_records,'sp_f1'):>8.4f}")
print(f"{'joint':<16} {avg(naive_records,'joint_em'):>8.4f} {avg(naive_records,'joint_f1'):>8.4f}")
print('=' * 52)
print(f"N: {len(naive_records)}")

# 分类型统计
print()
for qtype in ['bridge', 'comparison']:
    tr = [r for r in naive_records if r['type'] == qtype]
    print(f"{qtype} (n={len(tr)}):")
    print(f"  ans   EM={avg(tr,'em'):.3f}       F1={avg(tr,'f1'):.3f}")
    print(f"  sp    EM={avg(tr,'sp_em'):.3f}    F1={avg(tr,'sp_f1'):.3f}")
    print(f"  joint EM={avg(tr,'joint_em'):.3f} F1={avg(tr,'joint_f1'):.3f}")


Naive Baseline Result
Metrics                EM       F1
------------------------------------
ans                0.2300   0.2846
sp                 0.1500   0.5116
joint              0.0300   0.1479
N: 200

bridge (n=160):
  ans   EM=0.212       F1=0.278
  sp    EM=0.075    F1=0.470
  joint EM=0.013 F1=0.139
comparison (n=40):
  ans   EM=0.300       F1=0.312
  sp    EM=0.450    F1=0.677
  joint EM=0.100 F1=0.183


In [18]:

with open('naive_results.json', 'w') as f:
    json.dump({'summary': {
        'ans_em':    avg(naive_records,'em'),
        'ans_f1':    avg(naive_records,'f1'),
        'sp_em':     avg(naive_records,'sp_em'),
        'sp_f1':     avg(naive_records,'sp_f1'),
        'joint_em':  avg(naive_records,'joint_em'),
        'joint_f1':  avg(naive_records,'joint_f1'),
        'n':         len(naive_records)
    }, 'records': naive_records}, f, ensure_ascii=False, indent=2)

print('\nResult saved in naive_results.json')

# 答对/答错例子（同时展示sp情况）
correct = [r for r in naive_records if r['em'] == 1][:3]
wrong   = [r for r in naive_records if r['em'] == 0][:3]

print('\n【Examples of correct answers】')
for r in correct:
    print(f"  Q:  {r['question'][:65]}")
    print(f"  Prediction: {r['prediction']}  |  Gold answer: {r['answer']}")
    print(f"  sp: {r['sp_prediction']}")
    print(f"  sp_f1={r['sp_f1']:.2f}  joint_f1={r['joint_f1']:.2f}\n")

print('\n【Examples of incorrect answers】')
for r in wrong:
    print(f"  Q:  {r['question'][:65]}")
    print(f"  Prediction: {r['prediction']}  |  Gold answer: {r['answer']}")
    print(f"  sp: {r['sp_prediction']}")
    print(f"  sp_f1={r['sp_f1']:.2f}  joint_f1={r['joint_f1']:.2f}\n")


Result saved in naive_results.json

【Examples of correct answers】
  Q:  What is the name of the so-called reform opera for Vienna that ca
  Prediction: Alceste  |  Gold answer: "Alceste"
  sp: [['Orfeo ed Euridice', 2]]
  sp_f1=0.50  joint_f1=0.50

  Q:  The organization that Nicolae Titulescu served two terms as presi
  Prediction: 10 January 1920  |  Gold answer: 10 January 1920
  sp: [['League of Nations', 0]]
  sp_f1=0.67  joint_f1=0.67

  Q:  Which one of the Long Island Herald newspaper chain serves a vill
  Prediction: Nassau Herald  |  Gold answer: The Nassau Herald
  sp: [['Lawrence, Nassau County, New York', 1], ['Nassau Herald', 0]]
  sp_f1=0.80  joint_f1=0.80


【Examples of incorrect answers】
  Q:  Who voices Homer Simpson's character on the animated television s
  Prediction: Dan Castellaneta  |  Gold answer: Daniel Louis Castellaneta
  sp: []
  sp_f1=0.00  joint_f1=0.00

  Q:  What country of origin does  Dana Ivey and Two Weeks Notice have 
  Prediction:   |  Gold answe